In [1]:
from datasets import load_from_disk
from pathlib import Path
import os


/mnt/data/envs/finetune-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(os.listdir("../data"))
print(os.listdir("../data/processed"))


['format_for_sft.ipynb', 'raw', 'processed']
['control_por', 'control_spa', 'train', 'test', 'val']


In [3]:
train_ds = load_from_disk("../data/processed/train")
val_ds = load_from_disk("../data/processed/val")
test_ds = load_from_disk("../data/processed/test")
por_ds = load_from_disk("../data/processed/control_spa")
spa_ds = load_from_disk("../data/processed/control_por")


In [4]:
sample = test_ds[0]
for key, value in sample.items():
    print(key, ":", value)
print(test_ds.column_names)


inputs : Do you think the right answer to the question "what are safety goggles used to protect during experiments?" is "eye", given that
 safety goggles are used for protecting the retina during experiments?
targets : Yes
language : English
language_code : eng
annotation_type : re-annotations
user_id : 1736efafdce5010fdc24120fad1338ca8330b2f03d68088d472a217c7d67ea40
['inputs', 'targets', 'language', 'language_code', 'annotation_type', 'user_id']


### Formatting function

In [5]:
def format_example(example):
    return {
        "messages": [
            {"role": "user", "content": example["inputs"]},
            {"role": "assistant", "content": example["targets"]},
        ],
        "language_code": example["language_code"],
        "language": example["language"],
    }


In [6]:
original_columns = train_ds.column_names

formatted_train = train_ds.map(format_example, remove_columns=original_columns)
formatted_val = val_ds.map(format_example, remove_columns=original_columns)
formatted_test = test_ds.map(format_example, remove_columns=original_columns)
formatted_por_ds = por_ds.map(format_example, remove_columns=original_columns)
formatted_spa_ds = spa_ds.map(format_example, remove_columns=original_columns)


Map: 100%|██████████| 300/300 [00:00<00:00, 15395.90 examples/s]


In [7]:
for code in ["eng", "fra", "ita"]:
    subset = formatted_train.filter(lambda x: x["language_code"] == code)
    print(f"\n===== {code} =====")
    sample = subset[0]
    print(sample["messages"][0]["content"])
    print("-" * 40)
    print(sample["messages"][1]["content"])
    print("=" * 80)


Filter: 100%|██████████| 4882/4882 [00:00<00:00, 40349.32 examples/s]



===== eng =====
What initiatives are in place to promote sustainable tourism in Sri Lanka, balancing economic benefits with environmental conservation?
----------------------------------------
Sri Lanka has been actively promoting sustainable tourism by encouraging eco-friendly practices, community engagement, and wildlife conservation. Certification programs for eco-friendly accommodations and responsible tourism campaigns aim to ensure that tourism contributes positively to both the economy and the environment. Here are some key examples:

1. Policy and Regulatory Framework:

National Sustainable Tourism Policy 2018: This policy outlines a comprehensive framework for sustainable tourism development, including responsible visitor management, environmental protection, and community engagement.
National Ecotourism Strategy: This strategy focuses on promoting eco-friendly tourism practices and experiences, such as wildlife conservation and sustainable agriculture.
Regulations on plastic

Filter: 100%|██████████| 4882/4882 [00:00<00:00, 81381.62 examples/s]



===== fra =====
De quoi sont faits les anneaux de Saturne ?
----------------------------------------
Ils sont constitués de blocs de glace et de roches dont la taille varie entre celle d'un grain de sable et celle d'une montagne. Les anneaux de Saturne s'étendent sur plus de 500 000 kilomètres de large, mais sur moins de 1 kilomètre d'épaisseur. Beaucoup plus jeunes que Saturne elle-même, ils se seraient formés il y a quelques centaines de millions d'années, quand un ou plusieurs satellites auraient explosé, broyés par la forte gravité de la planète.


Filter: 100%|██████████| 4882/4882 [00:00<00:00, 62751.58 examples/s]


===== ita =====
Qual è la capitale d'Italia?
----------------------------------------
La capitale d'Italia è Roma.


In [8]:
print(formatted_train.column_names)
print(formatted_val.column_names)


['language', 'language_code', 'messages']
['language', 'language_code', 'messages']


In [9]:
formatted_train.save_to_disk("../data/processed/sft/train_sft")
formatted_val.save_to_disk("../data/processed/sft/validation_sft")
formatted_test.save_to_disk("../data/processed/sft/test_sft")
formatted_por_ds.save_to_disk("../data/processed/sft/probe_spa_sft")
formatted_spa_ds.save_to_disk("../data/processed/sft/probe_por_sft")


Saving the dataset (1/1 shards): 100%|██████████| 300/300 [00:00<00:00, 68400.26 examples/s]
